# NC-01: Rhino â†’ TopologicPy CellComplex
Imports OBJ exports from Rhino, assigns `room_type` / `door_type` dictionaries,
and builds a CellComplex with apertures ready for graph construction in NC-02.

In [62]:
import json
from collections import Counter
from pathlib import Path

from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster     import Cluster
from topologicpy.Dictionary  import Dictionary
from topologicpy.Topology    import Topology

BASE          = Path(r"E:\softwares-4\graph-ml\assign-04-node-classification")
GEOMETRY_PATH = BASE / "geometry"
GRAPHS_PATH   = BASE / "graphs"
GRAPHS_PATH.mkdir(exist_ok=True)

VALID_ROOM_TYPES = {
    "bedroom", "livingroom", "kitchen", "dining", "corridor",
    "stairs", "storeroom", "bathroom", "balcony"
}
VALID_DOOR_TYPES = {"passage", "door", "entrance_door"}

## 0. OBJ Checklist
Reads only the `g` group declaration lines from both OBJ files â€” the same as running
`grep "^g "` on each file. Prints a PASS / FAIL / NOTE checklist.

**Fix any FAIL in Rhino and re-export before running section 1.**

In [63]:
def parse_obj_groups(filepath):
    """Extract type name strings from OBJ group declaration lines."""
    groups = []
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("g "):
                parts = line.strip().split()
                groups.append(parts[-1])   # last token is always the type name
    return groups


def obj_checklist(rooms_path, apertures_path):
    room_groups = parse_obj_groups(rooms_path)
    door_groups = parse_obj_groups(apertures_path)

    checks = [
        ("rooms.obj found",          rooms_path.exists()),
        ("apertures.obj found",      apertures_path.exists()),
        ("rooms.obj has groups",     len(room_groups) > 0),
        ("apertures.obj has groups", len(door_groups) > 0),
    ]

    bad_rooms = [r for r in room_groups if r not in VALID_ROOM_TYPES]
    checks.append((f"room_type names valid  {room_groups}", len(bad_rooms) == 0))

    bad_doors = [d for d in door_groups if d not in VALID_DOOR_TYPES]
    checks.append((f"door_type names valid  {door_groups}", len(bad_doors) == 0))

    missing_rooms = sorted(VALID_ROOM_TYPES - set(room_groups))
    missing_doors = sorted(VALID_DOOR_TYPES - set(door_groups))

    print("=== OBJ Checklist ===")
    all_pass = True
    for label, ok in checks:
        status = "PASS" if ok else "FAIL"
        if not ok:
            all_pass = False
        print(f"  [{status}]  {label}")

    if missing_rooms:
        print(f"  [NOTE]  Room types not modelled: {missing_rooms}")
    if missing_doors:
        print(f"  [NOTE]  Door types not modelled: {missing_doors}")

    print("=" * 22)
    if all_pass:
        print("  Ready â€” proceed to section 1.\n")
    else:
        print("  Fix FAIL items in Rhino and re-export before continuing.\n")

    return all_pass, room_groups, door_groups


ok, room_groups, door_groups = obj_checklist(
    GEOMETRY_PATH / "rooms.obj",
    GEOMETRY_PATH / "apertures.obj"
)
assert ok, "Checklist failed â€” see FAIL items above."

=== OBJ Checklist ===
  [PASS]  rooms.obj found
  [PASS]  apertures.obj found
  [PASS]  rooms.obj has groups
  [PASS]  apertures.obj has groups
  [PASS]  room_type names valid  ['bedroom', 'livingroom', 'kitchen', 'bathroom', 'corridor', 'dining']
  [PASS]  door_type names valid  ['door', 'entrance_door']
  [NOTE]  Room types not modelled: ['balcony', 'stairs', 'storeroom']
  [NOTE]  Door types not modelled: ['passage']
  Ready â€” proceed to section 1.



## 1. Import room geometry
Each group is extracted into a temporary OBJ file and imported with ,
which is required for TopologicPy to produce topologically closed Cells (not Shells).
This is the same approach used in A01 - Primal Graph.

In [64]:
from topologicpy.Cell import Cell

objects = Topology.ByOBJPath(str(GEOMETRY_PATH / "rooms.obj"))
print("Objects returned:", len(objects))
for obj in objects:
    d = Topology.Dictionary(obj)
    print(" name:", Dictionary.ValueAtKey(d, "name"), " faces:", len(Topology.Faces(obj) or []))

Objects returned: 7
 name: default  faces: 0
 name: rooms bedroom  faces: 108
 name: rooms livingroom  faces: 36
 name: rooms kitchen  faces: 36
 name: rooms bathroom  faces: 56
 name: rooms corridor  faces: 56
 name: rooms dining  faces: 36


## 2. Extract cells and build selector vertices
One selector vertex per cell, carrying the `room_type` string as a dictionary value.

In [65]:
from topologicpy.Cell import Cell
from topologicpy.Vertex import Vertex
from collections import defaultdict

def faces_to_cells(faces, label):
    """Split faces into connected components (one per room), build a Cell each."""
    if not faces:
        return []
    n = len(faces)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    face_verts = []
    for f in faces:
        vset = set()
        for e in (Topology.Edges(f) or []):
            for v in (Topology.Vertices(e) or []):
                vset.add((round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2)))
        face_verts.append(vset)

    for i in range(n):
        for j in range(i + 1, n):
            if face_verts[i] & face_verts[j]:
                union(i, j)

    components = defaultdict(list)
    for i, f in enumerate(faces):
        components[find(i)].append(f)

    result = []
    for comp in components.values():
        c = Cell.ByFaces(comp)
        if c is None:
            print("    WARN Cell.ByFaces failed:", label, len(comp), "faces")
            continue
        c = Topology.RemoveCollinearEdges(c)
        result.append(c)
    return result


cells = []
selectors = []

for obj in objects:
    d = Topology.Dictionary(obj)
    name = Dictionary.ValueAtKey(d, "name") or ""
    room_type = name.strip().split()[-1] if name else ""
    if room_type not in VALID_ROOM_TYPES:
        print("  SKIP:", repr(name))
        continue
    faces = Topology.Faces(obj) or []
    room_cells = faces_to_cells(faces, room_type)
    for c in room_cells:
        s = Topology.InternalVertex(c)
        s = Topology.SetDictionary(s, Dictionary.ByKeyValue("room_type", room_type))
        selectors.append(s)
        cells.append(c)
    print("  ", room_type, "->", len(room_cells), "cell(s)")

print()
print("Total cells    :", len(cells))
print("Total selectors:", len(selectors))

  SKIP: 'default'
   bedroom -> 3 cell(s)
   livingroom -> 1 cell(s)
   kitchen -> 1 cell(s)
   bathroom -> 2 cell(s)
   corridor -> 2 cell(s)
   dining -> 1 cell(s)

Total cells    : 10
Total selectors: 10


## 3. Build CellComplex and transfer `room_type` dictionaries

In [66]:
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True)

cells_cc = Topology.Cells(cc)
faces_cc = Topology.Faces(cc)

# Count shared (internal) vs boundary faces
shared = []
boundary = []
for f in (faces_cc or []):
    adj = Topology.SuperTopologies(f, cc, topologyType="cell") or []
    if len(adj) >= 2:
        shared.append(f)
    else:
        boundary.append(f)

print("CellComplex built -", len(cells_cc), "cell(s)")
print("Total faces      :", len(faces_cc or []))
print("Shared faces     :", len(shared), " <- rooms share these walls")
print("Boundary faces   :", len(boundary))
print()
print("Room types assigned:")
for c in cells_cc:
    rt = Dictionary.ValueAtKey(Topology.Dictionary(c), "room_type")
    print("  " + str(rt))

CellComplex built - 10 cell(s)
Total faces      : 55
Shared faces     : 13  <- rooms share these walls
Boundary faces   : 42

Room types assigned:
  bedroom
  bathroom
  corridor
  bedroom
  dining
  livingroom
  kitchen
  corridor
  bathroom
  bedroom


## 4. Import apertures and assign `door_type`

In [67]:
aperture_objects = Topology.ByOBJPath(str(GEOMETRY_PATH / "apertures.obj"), selfMerge=True)
print("ByOBJPath returned", len(aperture_objects), "aperture object(s)")

aperture_faces = []
for obj in aperture_objects:
    d = Topology.Dictionary(obj)
    name = Dictionary.ValueAtKey(d, "name") or ""
    door_type = name.strip().split()[-1] if name else ""
    if door_type not in VALID_DOOR_TYPES:
        print("  SKIP:", repr(name))
        continue
    for f in (Topology.Faces(obj) or [obj]):
        f = Topology.SetDictionary(f, Dictionary.ByKeyValue("door_type", door_type))
        aperture_faces.append(f)
    print("  ", door_type, "->", len(Topology.Faces(obj) or [obj]), "face(s)")

print()
print("Total aperture faces:", len(aperture_faces))

ByOBJPath returned 3 aperture object(s)
  SKIP: 'default'
   door -> 9 face(s)
   entrance_door -> 3 face(s)

Total aperture faces: 12


## 5. Add apertures to CellComplex
Each aperture face must intersect a shared wall face to be attached.
If `Apertures attached: 0`, increase `tolerance` to `0.01` and re-run.

In [68]:
from topologicpy.Vertex import Vertex

def face_centroid(f):
    verts = Topology.Vertices(f) or []
    if not verts: return None
    x = sum(Vertex.X(v) for v in verts) / len(verts)
    y = sum(Vertex.Y(v) for v in verts) / len(verts)
    z = sum(Vertex.Z(v) for v in verts) / len(verts)
    return (round(x,2), round(y,2), round(z,2))

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** 0.5

cc_faces = Topology.Faces(cc) or []
cc_cents = [face_centroid(f) for f in cc_faces]
cc_cents = [c for c in cc_cents if c]
print("CellComplex has", len(cc_cents), "faces")
print("Aperture faces :", len(aperture_faces))
print()
for i, af in enumerate(aperture_faces):
    ac = face_centroid(af)
    if not ac:
        print("  aperture", i, ": no centroid")
        continue
    nearest_c = min(cc_cents, key=lambda c: dist3(ac, c))
    d = dist3(ac, nearest_c)
    dt = Dictionary.ValueAtKey(Topology.Dictionary(af), "door_type") or "?"
    print(dt + "[" + str(i) + "] centroid=" + str(ac) + "  nearest_face_dist=" + str(round(d, 2)))

CellComplex has 55 faces
Aperture faces : 12

door[0] centroid=(16310.0, -2850.0, 1170.16)  nearest_face_dist=330.02
door[1] centroid=(7763.91, 2240.0, 1022.6)  nearest_face_dist=1149.88
door[2] centroid=(7763.91, -760.0, 1022.6)  nearest_face_dist=1149.88
door[3] centroid=(7310.0, -2211.12, 1170.16)  nearest_face_dist=333.44
door[4] centroid=(14356.09, -3760.0, 1022.6)  nearest_face_dist=658.75
door[5] centroid=(17643.49, -3760.0, 1022.6)  nearest_face_dist=505.6
door[6] centroid=(7310.0, 1330.0, 1170.16)  nearest_face_dist=675.94
door[7] centroid=(10310.0, -2211.12, 1170.16)  nearest_face_dist=333.44
door[8] centroid=(19310.0, -2727.52, 1170.16)  nearest_face_dist=355.82
entrance_door[9] centroid=(13128.58, -760.0, 1022.6)  nearest_face_dist=510.71
entrance_door[10] centroid=(5511.99, 3892.07, 1015.0)  nearest_face_dist=508.28
entrance_door[11] centroid=(4310.0, -3558.09, 1015.0)  nearest_face_dist=525.35


In [69]:
cc = Topology.AddApertures(cc, aperture_faces,tolerance=1)

apers = Topology.Apertures(cc)
print("Apertures attached:", len(apers))

if len(apers) == 0:
    print("WARNING: No apertures attached.")
    print("  Verify door faces coincide with shared room walls in Rhino.")

print()
print("Door types attached:")
for a in apers:
    dt = Dictionary.ValueAtKey(Topology.Dictionary(a), "door_type")
    print("  " + str(dt))

Apertures attached: 12

Door types attached:
  door
  door
  door
  door
  door
  door
  door
  door
  door
  entrance_door
  entrance_door
  entrance_door


## 6. Pre-flight checklist
All checks must pass before moving to NC-02.

In [70]:
errors = []

if len(cells_cc) == 0:
    errors.append("CellComplex has no cells")

missing_room = [c for c in cells_cc
                if Dictionary.ValueAtKey(Topology.Dictionary(c), "room_type") is None]
if missing_room:
    errors.append(str(len(missing_room)) + " cell(s) missing room_type")

missing_door = [a for a in apers
                if Dictionary.ValueAtKey(Topology.Dictionary(a), "door_type") is None]
if missing_door:
    errors.append(str(len(missing_door)) + " aperture(s) missing door_type")

if len(apers) == 0:
    errors.append("No apertures attached - graph will have no edges")

print("--- Pre-flight ---")
if errors:
    for e in errors:
        print("  FAIL  " + e)
    print()
    print("Fix the issues above before running NC-02.")
else:
    print("  PASS  Ready for NC-02.")

--- Pre-flight ---
  PASS  Ready for NC-02.


## 7. Save summary
Writes `graphs/floor_plan_meta.json`. The `cc` variable stays in memory for NC-02.

In [71]:
room_counts = Counter(
    Dictionary.ValueAtKey(Topology.Dictionary(c), "room_type")
    for c in cells_cc
)
door_counts = Counter(
    Dictionary.ValueAtKey(Topology.Dictionary(a), "door_type")
    for a in apers
)

meta = {
    "cell_count":       len(cells_cc),
    "aperture_count":   len(apers),
    "room_type_counts": dict(room_counts),
    "door_type_counts": dict(door_counts),
    "notes": {
        "passage_present":    "passage" in door_counts,
        "missing_room_types": sorted(VALID_ROOM_TYPES - set(room_counts.keys())),
    }
}

meta_path = GRAPHS_PATH / "floor_plan_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Meta saved to {meta_path}")
print(json.dumps(meta, indent=2))
print("\nVariable 'cc' is ready â€” run NC-02 in the same session.")

Meta saved to E:\softwares-4\graph-ml\assign-04-node-classification\graphs\floor_plan_meta.json
{
  "cell_count": 10,
  "aperture_count": 12,
  "room_type_counts": {
    "bedroom": 3,
    "bathroom": 2,
    "corridor": 2,
    "dining": 1,
    "livingroom": 1,
    "kitchen": 1
  },
  "door_type_counts": {
    "door": 9,
    "entrance_door": 3
  },
  "notes": {
    "passage_present": false,
    "missing_room_types": [
      "balcony",
      "stairs",
      "storeroom"
    ]
  }
}

Variable 'cc' is ready â€” run NC-02 in the same session.
